# Phase 3 - Complete Data Preprocessing

This notebook prepares the verified phishing dataset for reproducible experiments. It performs data-quality validation, separates the target, creates the required stratified baseline split, checks for leakage, and prepares two preprocessing paths.

No model training, feature selection, zero-day claim, or unseen-domain claim is performed here.

## Phase 3 Introduction

The verified dataset contains 11,054 rows, 32 columns, 31 numeric input features, target `class`, no missing values, no duplicate rows, and no raw URL/domain identifier.

## Data Loading

The notebook loads `../dataset/raw/phishing.csv` when opened from the `notebooks` directory.

## Data Quality Validation

Shape, missing values, duplicates, and data types are checked. Duplicate rows are removed only if present.

## Target Separation

The target is explicitly identified as `class`; `X` contains the 31 non-target columns and `y` contains the labels.

## Train/Test Split

A reproducible stratified split uses `test_size=0.30`, `random_state=42`, and `stratify=y`.

## Class Distribution

Training and testing class counts and proportions are displayed and saved.

## Data Leakage Check

Train/test indices are checked for overlap. The SVM scaler is fitted only on `X_train`; no transformation is fitted on the complete dataset.

## Scaling Strategy

Path A retains numerical features unscaled for tree-based models. Path B uses `StandardScaler` in an sklearn `Pipeline` for a future SVM experiment.

## Processed Dataset Export

The cleaned dataset, train/test features and labels, scaled SVM features, and preprocessing metadata are saved under `dataset/processed/` and `results/`.

## Phase 3 Conclusion

This notebook does not train models or perform feature selection. The dataset does not contain URL/domain identifiers, so unseen-domain evaluation cannot be performed using this dataset. A random train/test split must not be described as zero-day detection.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = Path("../dataset/raw/phishing.csv") if Path.cwd().name == "notebooks" else PROJECT_ROOT / "dataset/raw/phishing.csv"
RAW_PATH = RAW_PATH.resolve()
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Place the dataset at {RAW_PATH}")

df = pd.read_csv(RAW_PATH)
print(f"Loaded: {RAW_PATH}")
print(f"Initial shape: {df.shape}")
display(df.head())

# Data quality validation. Duplicate removal is conditional and no values are fabricated.
print("Shape:", df.shape)
print("Total missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Data types:")
display(df.dtypes.rename("dtype").to_frame())

duplicate_count = int(df.duplicated().sum())
if duplicate_count:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Removed {duplicate_count} duplicate rows.")
else:
    print("No duplicate rows were present; no rows were removed.")

missing_count = int(df.isna().sum().sum())
if missing_count:
    raise ValueError("Unexpected missing values found. Choose an explicit training-only imputation strategy before proceeding.")

assert df.shape == (11054, 32), f"Verified dataset shape changed unexpectedly: {df.shape}"
assert duplicate_count == 0, "The verified dataset should not contain duplicate rows."
assert missing_count == 0, "The verified dataset should not contain missing values."
assert all(pd.api.types.is_numeric_dtype(df[column]) for column in df.columns), "All verified columns should be numeric."

# Target separation and label validation.
TARGET_COLUMN = "class"
if TARGET_COLUMN not in df.columns:
    raise KeyError(f"Expected target column {TARGET_COLUMN!r} was not found.")

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN].copy()
EXPECTED_LABELS = {-1, 1}
observed_labels = set(y.unique())
if observed_labels != EXPECTED_LABELS:
    raise ValueError(f"Expected class labels {EXPECTED_LABELS}, found {observed_labels}")

print("Target column:", TARGET_COLUMN)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Observed class labels:", sorted(observed_labels))
print("Overall class distribution:")
display(y.value_counts().sort_index().rename("count").to_frame())

# Required reproducible stratified 70/30 baseline split.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
assert X_train.shape == (7737, 31)
assert X_test.shape == (3317, 31)
assert y_train.shape == (7737,)
assert y_test.shape == (3317,)

# Class distributions are persisted for reproducible reporting.
train_distribution = pd.DataFrame({
    "count": y_train.value_counts().sort_index(),
    "proportion": y_train.value_counts(normalize=True).sort_index(),
})
test_distribution = pd.DataFrame({
    "count": y_test.value_counts().sort_index(),
    "proportion": y_test.value_counts(normalize=True).sort_index(),
})
print("Training class distribution:")
display(train_distribution)
print("Testing class distribution:")
display(test_distribution)
train_distribution.add_prefix("train_").join(test_distribution.add_prefix("test_")).to_csv(
    RESULTS_DIR / "preprocessing_class_distribution.csv"
)

# Leakage checks: split indices are disjoint and transformations are train-only.
overlap = X_train.index.intersection(X_test.index)
assert len(overlap) == 0, f"Train/test index overlap detected: {list(overlap)}"
print("Train/test indices overlap:", len(overlap))
print("Index leakage check: passed")
print("Any fitted preprocessing object must be fit on X_train only.")
print("No scaler is fit on X, the complete dataset, or X_test.")

# Path A: tree-based models retain the numeric features without scaling.
X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

# Path B: a future SVM experiment can use this train-only fitted scaler pipeline.
svm_preprocessor = Pipeline([("standard_scaler", StandardScaler())])
X_train_svm = svm_preprocessor.fit_transform(X_train)
X_test_svm = svm_preprocessor.transform(X_test)
assert np.allclose(X_train_svm.mean(axis=0), 0.0, atol=1e-10)
assert np.allclose(X_train_svm.std(axis=0), 1.0, atol=1e-10)
print("Path A tree features:", X_train_tree.shape, "(unscaled)")
print("Path B SVM features:", X_train_svm.shape, "(train-only scaled pipeline)")
print("No model has been trained.")

# Export cleaned data and reproducible experiment artifacts.
cleaned_path = PROCESSED_DIR / "phishing_clean.csv"
df.to_csv(cleaned_path, index=False)
X_train_tree.to_csv(PROCESSED_DIR / "X_train_tree.csv", index=False)
X_test_tree.to_csv(PROCESSED_DIR / "X_test_tree.csv", index=False)
y_train.to_frame(name=TARGET_COLUMN).to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_frame(name=TARGET_COLUMN).to_csv(PROCESSED_DIR / "y_test.csv", index=False)
pd.DataFrame(X_train_svm, columns=X_train.columns).to_csv(PROCESSED_DIR / "X_train_svm_scaled.csv", index=False)
pd.DataFrame(X_test_svm, columns=X_test.columns).to_csv(PROCESSED_DIR / "X_test_svm_scaled.csv", index=False)

preprocessing_info = {
    "raw_path": str(RAW_PATH.relative_to(PROJECT_ROOT)),
    "cleaned_shape": list(df.shape),
    "target_column": TARGET_COLUMN,
    "input_feature_count": int(X.shape[1]),
    "duplicate_rows_removed": duplicate_count,
    "missing_values_found": missing_count,
    "split": {"test_size": 0.30, "random_state": 42, "stratified": True},
    "train_shape": [int(X_train.shape[0]), int(X_train.shape[1])],
    "test_shape": [int(X_test.shape[0]), int(X_test.shape[1])],
    "tree_path": "numeric features retained without scaling",
    "svm_path": "StandardScaler fitted only on X_train through sklearn Pipeline",
    "model_training_performed": False,
    "feature_selection_performed": False,
    "unseen_domain_evaluation_supported": False,
    "unseen_domain_limitation": "The dataset does not contain URL/domain identifiers, so unseen-domain evaluation cannot be performed using this dataset.",
}
(RESULTS_DIR / "preprocessing_info.json").write_text(json.dumps(preprocessing_info, indent=2), encoding="utf-8")
print("Saved cleaned dataset:", cleaned_path)
print("Saved reproducible train/test artifacts in:", PROCESSED_DIR)
print("Saved preprocessing metadata in:", RESULTS_DIR / "preprocessing_info.json")

Loaded: ./dataset\raw\phishing.csv
Initial shape: (11054, 32)


,Index,UsingIP,LongURL,ShortURL,Symbol@,Redirecting//,PrefixSuffix-,SubDomains,HTTPS,DomainRegLen,...,UsingPopupWindow,IframeRedirection,AgeofDomain,DNSRecording,WebsiteTraffic,PageRank,GoogleIndex,LinksPointingToPage,StatsReport,class
0,0,1,1,1,1,1,-1,0,1,-1,...,1,1,-1,-1,0,-1,1,1,1,-1
1,1,1,0,1,1,1,-1,-1,-1,-1,...,1,1,1,-1,1,-1,1,0,-1,-1
2,2,1,0,1,1,1,-1,-1,-1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
3,3,1,0,-1,1,1,-1,1,1,-1,...,-1,1,-1,-1,0,-1,1,1,1,1
4,4,-1,0,-1,1,-1,-1,1,1,-1,...,1,1,1,1,1,-1,1,-1,-1,1


Shape: (11054, 32)
Total missing values: 0
Duplicate rows: 0
Data types:


,dtype
Index,int64
UsingIP,int64
LongURL,int64
ShortURL,int64
Symbol@,int64
Redirecting//,int64
PrefixSuffix-,int64
SubDomains,int64
HTTPS,int64
DomainRegLen,int64


No duplicate rows were present; no rows were removed.
Target column: class
X shape: (11054, 31)
y shape: (11054,)
Observed class labels: [np.int64(-1), np.int64(1)]
Overall class distribution:


...

,count,proportion
class,,
-1,3428,0.443066
1,4309,0.556934


Testing class distribution:


,count,proportion
class,,
-1,1469,0.44287
1,1848,0.55713


Train/test indices overlap: 0
Index leakage check: passed
Any fitted preprocessing object must be fit on X_train only.
No scaler is fit on X, the complete dataset, or X_test.
Path A tree features: (7737, 31) (unscaled)
Path B SVM features: (7737, 31) (train-only scaled pipeline)
No model has been trained.


Saved cleaned dataset: ./dataset\processed\phishing_clean.csv
Saved reproducible train/test artifacts in: ./dataset\processed
Saved preprocessing metadata in: ./results\preprocessing_info.json


In [2]:
print("Phase 3 summary")
print(f"- Cleaned dataset shape: {df.shape}")
print(f"- Target: {TARGET_COLUMN}; input features: {X.shape[1]}")
print(f"- X_train: {X_train.shape}; X_test: {X_test.shape}")
print(f"- y_train: {y_train.shape}; y_test: {y_test.shape}")
print("- Split: test_size=0.30, random_state=42, stratify=y")
print("- Leakage checks passed; SVM scaler fitted on X_train only")
print("- Model training and feature selection: not performed")
print("- Unseen-domain evaluation: unavailable because no URL/domain identifiers exist")

Phase 3 summary
- Cleaned dataset shape: (11054, 32)
- Target: class; input features: 31
- X_train: (7737, 31); X_test: (3317, 31)
- y_train: (7737,); y_test: (3317,)
- Split: test_size=0.30, random_state=42, stratify=y
- Leakage checks passed; SVM scaler fitted on X_train only
- Model training and feature selection: not performed
- Unseen-domain evaluation: unavailable because no URL/domain identifiers exist
